# 01 · Motor de enlaces (URL)

**Fase 2 — Semanas 3 a 8, en paralelo con el corpus.**

Este motor **no necesita el corpus**: las URL no tienen idioma. Se entrena con datos públicos.

En este cuaderno hacemos la **línea base clásica** del PDF: n-gramas de caracteres + Regresión Logística y Naive Bayes. El modelo profundo (CNN) va en Colab (necesita GPU).

**Antes de correr:** descarga `malicious_phish.csv` de Kaggle y colócalo en `data/crudo/`.
Si no está, el cuaderno usa un DEMO pequeño para poder correr igual.

**Entregable:** este cuaderno, dos modelos guardados (`modelos/`) y una tabla comparativa de métricas.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42                 # semilla fija = resultados reproducibles
MAX_FILAS = 60_000        # muestra para correr rápido en local; en Colab sube o pon None
np.random.seed(SEED)

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CRUDO = RAIZ / 'data' / 'crudo'
PARTICIONES = RAIZ / 'data' / 'particiones'
MODELOS = RAIZ / 'modelos'
RESULTADOS = RAIZ / 'resultados'
for d in (PARTICIONES, MODELOS, RESULTADOS):
    d.mkdir(exist_ok=True)
print('Proyecto:', RAIZ)

In [ ]:
# --- Carga: dataset real de data/crudo/ si existe; si no, DEMO ---
def detectar_columnas(df):
    col_url = next((c for c in df.columns if c.lower() in ('url', 'urls', 'link')), df.columns[0])
    col_lab = next((c for c in df.columns if c.lower() in ('type', 'label', 'result', 'class')), df.columns[-1])
    return col_url, col_lab

def a_binario(v):
    # benign/legítimo -> legitimo ; phishing/malware/defacement/1 -> fraudulento
    s = str(v).strip().lower()
    return 'legitimo' if s in ('benign', 'legitimate', 'good', '0', 'legitimo') else 'fraudulento'

archivo = CRUDO / 'malicious_phish.csv'
if not archivo.exists():
    otros = list(CRUDO.glob('*.csv'))
    archivo = otros[0] if otros else None

if archivo is not None:
    print('Usando dataset real:', archivo.name)
    df = pd.read_csv(archivo)
    col_url, col_lab = detectar_columnas(df)
    df = df[[col_url, col_lab]].rename(columns={col_url: 'url', col_lab: 'etiqueta_orig'})
    df['etiqueta'] = df['etiqueta_orig'].map(a_binario)
else:
    print('No hay CSV en data/crudo/ -> generando DEMO (placeholder).')
    legit = [f'https://www.sitio{i}.com/pagina/articulo-{i}' for i in range(300)]
    fraude = [f'http://secure-login-banco{i}.xyz/verify?acc={i}' for i in range(300)]
    df = pd.DataFrame({'url': legit + fraude,
                       'etiqueta': ['legitimo']*300 + ['fraudulento']*300})

df = df.dropna(subset=['url']).drop_duplicates(subset=['url']).reset_index(drop=True)
print('Total tras limpiar duplicados:', len(df))
df.head()

In [ ]:
# Balance de clases
print(df['etiqueta'].value_counts())
print()
print(df['etiqueta'].value_counts(normalize=True).round(3))

In [ ]:
from sklearn.model_selection import train_test_split

# Muestra estratificada para correr rápido en local (conserva la proporción de clases)
if MAX_FILAS and len(df) > MAX_FILAS:
    df, _ = train_test_split(df, train_size=MAX_FILAS, stratify=df['etiqueta'], random_state=SEED)
    df = df.reset_index(drop=True)
    print('Muestra usada:', len(df))

# Particiones 70 / 15 / 15, estratificadas y con semilla fija
train, resto = train_test_split(df, test_size=0.30, stratify=df['etiqueta'], random_state=SEED)
val, test = train_test_split(resto, test_size=0.50, stratify=resto['etiqueta'], random_state=SEED)

for nombre, part in [('url_train', train), ('url_val', val), ('url_test', test)]:
    part.to_csv(PARTICIONES / f'{nombre}.csv', index=False)
print(f'train={len(train)}  val={len(val)}  test={len(test)}')
print('Particiones guardadas en data/particiones/')

In [ ]:
# Línea base sobre URL: n-gramas de CARACTERES, no de palabras.
# Una URL no tiene palabras, tiene subcadenas sospechosas.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score, confusion_matrix

X_tr, y_tr = train['url'], train['etiqueta']
X_val, y_val = val['url'], val['etiqueta']

def vectorizador():
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)

logreg = make_pipeline(vectorizador(),
                       LogisticRegression(max_iter=1000, class_weight='balanced'))
nb = make_pipeline(vectorizador(), MultinomialNB())

logreg.fit(X_tr, y_tr)
nb.fit(X_tr, y_tr)
print('Modelos entrenados.')

In [ ]:
# Evaluación en VALIDACIÓN (el test NO se toca hasta la Fase 4)
for nombre, modelo in [('Regresión Logística', logreg), ('Naive Bayes', nb)]:
    pred = modelo.predict(X_val)
    f1 = f1_score(y_val, pred, pos_label='fraudulento', zero_division=0)
    print(f'===== {nombre}  |  F1 (fraudulento) = {f1:.3f} =====')
    print(classification_report(y_val, pred, digits=3, zero_division=0))
    print()

In [ ]:
# Matriz de confusión del mejor baseline (Regresión Logística)
import matplotlib.pyplot as plt
import seaborn as sns

clases = ['legitimo', 'fraudulento']
cm = confusion_matrix(y_val, logreg.predict(X_val), labels=clases)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases, cbar=False, ax=ax)
ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real')
ax.set_title('Motor URL — Regresión Logística (validación)')
plt.tight_layout()
plt.savefig(RESULTADOS / '01_url_matriz_logreg.png', dpi=120)
plt.show()

In [ ]:
# Guardar modelos y predicciones de validación (se reusan en la Fase 4)
import joblib
joblib.dump(logreg, MODELOS / 'url_logreg.joblib')
joblib.dump(nb, MODELOS / 'url_nb.joblib')

pd.DataFrame({'url': X_val, 'y_real': y_val,
              'pred_logreg': logreg.predict(X_val),
              'pred_nb': nb.predict(X_val)}).to_csv(RESULTADOS / '01_url_val_pred.csv', index=False)
print('Modelos en modelos/ y predicciones en resultados/ ✔')

## Qué sigue

1. **Modelo profundo (CNN de caracteres)** sobre la URL — en Colab con GPU. Se compara contra estos baselines.
2. Escalar: subir `MAX_FILAS` (o ponerlo en `None`) al correr en Colab con el dataset completo.
3. El **test** queda intacto hasta la Fase 4 (`03_evaluacion.ipynb`), donde se aplica McNemar entre el modelo profundo y estos baselines.

> Disciplina: fijamos semilla, guardamos modelos **y predicciones**, y no miramos el test.